In [4]:
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.linear_model import Lasso
from sklearn.model_selection import KFold, cross_val_score

import pandas as pd
import numpy as np
import optuna

import os

In [5]:
data = pd.read_csv('../data/processed/train.csv')

X_train = data.drop(columns=['LogSalePrice'])
y_train = data['LogSalePrice']

In [6]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# XGBoost

In [7]:
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 2000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.2, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }
    
    model = XGBRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )
    
    scores = cross_val_score(
        model, 
        X_train, 
        y_train, 
        cv=cv, 
        scoring='neg_root_mean_squared_error'
    )

    return -scores.mean()

In [8]:
study_xgb = optuna.create_study(
    direction='minimize',  # минимизируем RMSE
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_xgb.optimize(objective_xgb, n_trials=50)
print('Best RMSE: ', study_xgb.best_value)
print('Best params: ', study_xgb.best_params)

[I 2026-10-26 17:21:49,639] A new study created in memory with name: no-name-9d4dadd5-8d7a-4bcf-bb87-bad9a611f164
[I 2026-10-26 17:21:59,856] Trial 0 finished with value: 0.14476862794054746 and parameters: {'n_estimators': 900, 'learning_rate': 0.0862735828664018, 'max_depth': 5, 'min_child_weight': 6, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.32479561626896214, 'reg_alpha': 2.9152036385288193e-08, 'reg_lambda': 0.6245760287469893}. Best is trial 0 with value: 0.14476862794054746.
[I 2026-10-26 17:22:11,944] Trial 1 finished with value: 0.13716400917899949 and parameters: {'n_estimators': 1300, 'learning_rate': 0.04170553216181044, 'max_depth': 2, 'min_child_weight': 10, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.36987128854262097, 'reg_alpha': 2.8483918709107956e-07, 'reg_lambda': 4.4734294104626844e-07}. Best is trial 1 with value: 0.13716400917899949.
[I 2026-10-26 17:22:20,404] Trial 2 finished with value: 0.13430261411320069 and parameters: {'n_estimators

Best RMSE:  0.1320103075615242
Best params:  {'n_estimators': 700, 'learning_rate': 0.0395322210882127, 'max_depth': 3, 'min_child_weight': 2, 'subsample': 0.5656165865903986, 'colsample_bytree': 0.26820212238059554, 'reg_alpha': 0.1525750636114867, 'reg_lambda': 0.0001221134496184683}


# CatBoost

In [9]:
def objective_catboost(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 2000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 20.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 0.0, 2.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
    }
    
    model = CatBoostRegressor(
        **params,
        random_state=42,
        verbose=0,                 # без лога каждой итерации
        allow_writing_files=False  # не создавать папку catboost_info
    )
    
    scores = cross_val_score(
        model, 
        X_train, 
        y_train, 
        cv=cv, 
        scoring='neg_root_mean_squared_error'
    )
    
    return -scores.mean()

In [10]:
study_catboost = optuna.create_study(
    direction='minimize',  # минимизируем RMSE
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_catboost.optimize(objective_catboost, n_trials=50)
print('Best RMSE: ', study_catboost.best_value)
print('Best params: ', study_catboost.best_params)

[I 2026-10-26 17:29:22,449] A new study created in memory with name: no-name-1f3d9c42-6be9-4025-b4fb-6a9232c41030
[I 2026-10-26 17:29:41,593] Trial 0 finished with value: 0.1328015164036083 and parameters: {'iterations': 1000, 'learning_rate': 0.08927180304353628, 'depth': 7, 'l2_leaf_reg': 6.009974718380312, 'random_strength': 0.31203728088487304, 'bagging_temperature': 0.15599452033620265}. Best is trial 0 with value: 0.1328015164036083.
[I 2026-10-26 17:29:51,374] Trial 1 finished with value: 0.13465541620113913 and parameters: {'iterations': 500, 'learning_rate': 0.07348118405270448, 'depth': 7, 'l2_leaf_reg': 8.341106432362084, 'random_strength': 0.041168988591604894, 'bagging_temperature': 0.9699098521619943}. Best is trial 0 with value: 0.1328015164036083.
[I 2026-10-26 17:30:05,432] Trial 2 finished with value: 0.13353573314231504 and parameters: {'iterations': 1800, 'learning_rate': 0.016305687346221478, 'depth': 4, 'l2_leaf_reg': 1.7322667470546256, 'random_strength': 0.60848

Best RMSE:  0.13199982368121335
Best params:  {'iterations': 500, 'learning_rate': 0.07338309364174002, 'depth': 6, 'l2_leaf_reg': 3.6840597426557835, 'random_strength': 0.5864550589158318, 'bagging_temperature': 0.6592690836357396}


# Lasso

In [11]:
def objective_lasso(trial):
    params = {
        'alpha': trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
    }
    
    model = Lasso(
        **params,
        max_iter=50000,  # чтобы при маленьких alpha модель успевала сойтись
        random_state=42
    )
    
    scores = cross_val_score(
        model, 
        X_train, 
        y_train, 
        cv=cv, 
        scoring='neg_root_mean_squared_error'
    )
    
    return -scores.mean()

In [12]:
study_lasso = optuna.create_study(
    direction='minimize',  # минимизируем RMSE
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_lasso.optimize(objective_lasso, n_trials=50)
print('Best RMSE: ', study_lasso.best_value)
print('Best params: ', study_lasso.best_params)

[I 2026-10-26 17:36:34,374] A new study created in memory with name: no-name-e72afb17-c272-4846-b41a-f0df81d7e003
[I 2026-10-26 17:36:34,573] Trial 0 finished with value: 0.13572867839825004 and parameters: {'alpha': 0.0001329291894316216}. Best is trial 0 with value: 0.13572867839825004.
[I 2026-10-26 17:36:34,675] Trial 1 finished with value: 0.14979271228984367 and parameters: {'alpha': 0.0071144760093434225}. Best is trial 0 with value: 0.13572867839825004.
[I 2026-10-26 17:36:34,789] Trial 2 finished with value: 0.14081743766140828 and parameters: {'alpha': 0.001570297088405539}. Best is trial 0 with value: 0.13572867839825004.
[I 2026-10-26 17:36:34,919] Trial 3 finished with value: 0.13525063226958672 and parameters: {'alpha': 0.0006251373574521745}. Best is trial 3 with value: 0.13525063226958672.
[I 2026-10-26 17:36:35,168] Trial 4 finished with value: 0.13755047596557976 and parameters: {'alpha': 2.9380279387035334e-05}. Best is trial 3 with value: 0.13525063226958672.
[I 202

Best RMSE:  0.13446674956717372
Best params:  {'alpha': 0.0004050358187113016}


In [13]:
res = pd.DataFrame(
    {'model_name': ['XGBoost', 'CatBoost', 'Lasso'],
     'rmse': [study_xgb.best_value, study_catboost.best_value, study_lasso.best_value]}
)
res.sort_values('rmse')


,model_name,rmse
1,CatBoost,0.132000
0,XGBoost,0.132010
2,Lasso,0.134467


# Final

In [14]:
test_data = pd.read_csv('../data/processed/test.csv')
test_raw = pd.read_csv('../data/raw/test.csv')

X = test_data.drop(columns=['Id'])

In [15]:
model = CatBoostRegressor(
    **study_catboost.best_params,
    random_state=42,
    verbose=0,                 # без лога каждой итерации
    allow_writing_files=False  # не создавать папку catboost_info
)

model.fit(X_train, y_train)

predictions = model.predict(X)
final_predictions = np.expm1(predictions)

In [16]:
submission = pd.DataFrame({
    'Id': test_raw['Id'],
    'SalePrice': final_predictions
})
submission

,Id,SalePrice
0,1461,125370.243932
1,1462,155457.205061
2,1463,182399.383351
3,1464,188579.588800
4,1465,188714.918336
...,...,...
1454,2915,76894.888209
1455,2916,80429.901302
1456,2917,153355.746548
1457,2918,114161.865379


In [17]:
os.makedirs('../submissions', exist_ok=True)

submission.to_csv('../submissions/submissioncatboost.csv', index=False)

In [18]:
results = pd.DataFrame({
    'Name': ['CatBoost', 'XGBoost', 'Lasso'],
    'CV': [study_catboost.best_value, study_xgb.best_value, study_lasso.best_value],
    'LB': [0.12901, None, None]
})

results

,Name,CV,LB
0,CatBoost,0.132000,0.12901
1,XGBoost,0.132010,NaN
2,Lasso,0.134467,NaN
